# 07 — Tutorial 2: Biased Simulation of Protein Unfolding in GROMACS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/Module%202%20-%20Molecular%20dynamics/notebooks/07_tutorial_protein_unfolding_gromacs.ipynb)

## 🎯 Learning Objectives

- Understand the GROMACS simulation workflow for proteins
- Select and apply the AMBER99SB-ILDN force field with TIP3P water
- Set up steered MD (SMD) pulling simulations to unfold a protein
- Understand and apply umbrella sampling along a reaction coordinate
- Reconstruct the potential of mean force (PMF) using the Weighted Histogram Analysis Method (WHAM)
- Apply the Jarzynski equality to estimate free energy from non-equilibrium work
- Parse and analyze GROMACS output files with Python

## 1. Introduction: Protein Unfolding and Chignolin

### 1.1 Why Simulate Protein Unfolding?

Protein folding/unfolding is fundamental to biology — misfolding causes Alzheimer's, Parkinson's, and prion diseases. Free energy differences between folded (F) and unfolded (U) states govern thermodynamic stability:

$$\Delta G_{\text{fold}} = G_F - G_U = -k_B T \ln K_{eq}$$

Direct MD observation of unfolding at physiological conditions is rare because the timescale ($\mu$s–ms) far exceeds accessible MD simulation times (ns–$\mu$s). We use **biased simulation methods** to accelerate sampling.

### 1.2 Chignolin: A Minimal Protein

**Chignolin** (CLN025) is a 10-residue β-hairpin peptide:
```
Sequence: Tyr-Tyr-Asp-Pro-Glu-Thr-Gly-Thr-Trp-Tyr
PDB ID:   2RVD  (folded, β-hairpin)
```

| Property | Value |
|----------|-------|
| Residues | 10 |
| Atoms (all-atom) | ~166 protein + ~thousands of water |
| $T_m$ | ~340 K (thermo-stable) |
| $\Delta G_{fold}$ | ~−5 kcal/mol |
| Folding time | ~0.8 μs |

Chignolin is ideal for tutorials: small enough for fast simulation, large enough to exhibit folding/unfolding transitions.

In [ ]:
# =============================================================================
# Ch121a: Molecular Dynamics — Notebook 07: Protein Unfolding Tutorial (GROMACS)
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy import stats, optimize, special
from scipy.signal import savgol_filter

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.labelsize': 12, 'legend.fontsize': 10})
print('Packages loaded.')

## 2. GROMACS File Types and Workflow

### 2.1 Key File Types

| Extension | Name | Content |
|-----------|------|---------|
| `.pdb` | Protein Data Bank | Initial 3D coordinates |
| `.gro` | GROMOS coordinate | Coordinates + box (GROMACS format) |
| `.top` | Topology | Bonds, angles, dihedrals, FF parameters |
| `.itp` | Include topology | Modular topology fragment |
| `.mdp` | MD parameters | Simulation settings (steps, T, P, ...) |
| `.tpr` | Run input | Compiled binary: structure + topology + params |
| `.xtc` | Trajectory (compressed) | Coordinates vs. time |
| `.trr` | Trajectory (full) | Coords + velocities + forces |
| `.edr` | Energy file (binary) | All thermodynamic properties vs. time |
| `.xvg` | XVG data | ASCII data for plotting (xmgrace format) |

### 2.2 Complete Workflow

```
protein.pdb
     │
     ▼ gmx pdb2gmx  (assign FF, generate topology)
protein.gro + topol.top + posre.itp
     │
     ▼ gmx editconf  (center, define box)
protein_box.gro
     │
     ▼ gmx solvate   (fill with water)
protein_solv.gro  (updated topol.top)
     │
     ▼ gmx genion    (add Na⁺/Cl⁻ to neutralize)
protein_ions.gro  (updated topol.top)
     │
     ▼ gmx grompp + gmx mdrun  (energy minimization)
     │
     ▼ gmx grompp + gmx mdrun  (NVT equilibration, 100 ps)
     │
     ▼ gmx grompp + gmx mdrun  (NPT equilibration, 100 ps)
     │
     ▼ gmx grompp + gmx mdrun  (SMD pulling, 5 ns)
pull_coord.xvg  pull_force.xvg
     │
     ▼ Extract umbrella windows from pulling traj
     │
     ▼ gmx mdrun  (umbrella sampling, N windows × 5 ns each)
     │
     ▼ gmx wham   → pmf.xvg  (PMF / free energy profile)
```

## 3. Force Field: AMBER99SB-ILDN + TIP3P

**AMBER99SB-ILDN** (Lindorff-Larsen *et al.* 2010) is an improved version of AMBER99SB with better side-chain dihedral parameters for Ile, Leu, Asp, Asn.

| Feature | Description |
|---------|-------------|
| Backbone dihedrals | φ/ψ from NMR + QM |
| Side-chain χ dihedrals | Refitted to NMR scalar couplings |
| Charges | RESP from HF/6-31G* |
| Water model | TIP3P (3-site rigid) |

**TIP3P water:**

| Parameter | Value |
|-----------|-------|
| O-H bond | 0.9572 Å |
| H-O-H angle | 104.52° |
| $q_O$ | −0.834 e |
| $q_H$ | +0.417 e |
| $\varepsilon_{OO}$ | 0.6364 kJ/mol |
| $\sigma_{OO}$ | 3.15061 Å |

In GROMACS, select the force field with:
```bash
gmx pdb2gmx -f protein.pdb -o protein.gro -water tip3p
# → select AMBER99SB-ILDN from the menu
```

## 4. MDP Files for Each Simulation Stage

The `.mdp` file controls every aspect of a GROMACS run.

In [ ]:
mdp_em = """
; ============================================================
; em.mdp — Steepest-descent energy minimization
; ============================================================
integrator      = steep          ; Steepest descents
emtol           = 1000.0         ; Stop when Fmax < 1000 kJ/mol/nm
emstep          = 0.01           ; Initial step size (nm)
nsteps          = 50000

; Neighbor searching
cutoff-scheme   = Verlet
nstlist         = 1
rcoulomb        = 1.0            ; nm
rvdw            = 1.0            ; nm

; Electrostatics
coulombtype     = PME
pme-order       = 4
fourierspacing  = 0.16

; Constraints
constraints     = h-bonds
"""

mdp_nvt = """
; ============================================================
; nvt.mdp — NVT equilibration (100 ps, 300 K)
; ============================================================
integrator      = md
nsteps          = 50000          ; 50000 × 2 fs = 100 ps
dt              = 0.002          ; 2 fs

nstenergy       = 500
nstlog          = 500
nstxout-compressed = 500

cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0

coulombtype     = PME
pme-order       = 4
fourierspacing  = 0.16

tcoupl          = V-rescale      ; Velocity-rescaling thermostat
tc-grps         = Protein  Non-Protein
tau_t           = 0.1    0.1    ; coupling time (ps)
ref_t           = 300    300    ; target temperature (K)

pcoupl          = no             ; no pressure coupling in NVT

constraints     = h-bonds
constraint-algorithm = LINCS

; Position restraints on protein heavy atoms
define          = -DPOSRES
"""

mdp_npt = """
; ============================================================
; npt.mdp — NPT equilibration (100 ps, 300 K, 1 bar)
; ============================================================
integrator      = md
nsteps          = 50000
dt              = 0.002

nstenergy       = 500
nstlog          = 500
nstxout-compressed = 500

cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0

coulombtype     = PME
pme-order       = 4
fourierspacing  = 0.16

tcoupl          = V-rescale
tc-grps         = Protein  Non-Protein
tau_t           = 0.1    0.1
ref_t           = 300    300

pcoupl          = Parrinello-Rahman
pcoupltype      = isotropic
tau_p           = 2.0            ; ps
ref_p           = 1.0            ; bar
compressibility = 4.5e-5         ; bar⁻¹ (water)

constraints     = h-bonds
constraint-algorithm = LINCS

define          = -DPOSRES
"""

print('MDP files prepared:')
for name, content in [('em.mdp', mdp_em), ('nvt.mdp', mdp_nvt), ('npt.mdp', mdp_npt)]:
    with open(f'/tmp/{name}', 'w') as f:
        f.write(content)
    print(f'  /tmp/{name} ({len(content.splitlines())} lines)')

In [ ]:
mdp_pull = """
; ============================================================
; pull.mdp — Steered MD pulling simulation (5 ns)
; ============================================================
integrator      = md
nsteps          = 2500000        ; 5 ns at dt=2 fs
dt              = 0.002

nstenergy       = 1000
nstlog          = 1000
nstxout-compressed = 500         ; save coords every 1 ps

cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0

coulombtype     = PME
pme-order       = 4
fourierspacing  = 0.16

tcoupl          = V-rescale
tc-grps         = Protein  Non-Protein
tau_t           = 0.1    0.1
ref_t           = 300    300

pcoupl          = Parrinello-Rahman
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5

constraints     = h-bonds
constraint-algorithm = LINCS

; ===========================================================
; PULLING PARAMETERS (SMD: constant velocity)
; ===========================================================
pull            = yes
pull-ngroups    = 2
pull-ncoords    = 1

; Group 1: N-terminus (first residue Cα)
pull-group1-name = N_term
; Group 2: C-terminus (last residue Cα)
pull-group2-name = C_term

pull-coord1-type      = umbrella  ; harmonic bias
pull-coord1-geometry  = distance  ; pull along end-to-end distance
pull-coord1-groups    = 1 2
pull-coord1-dim       = Y Y Y     ; pull in all directions
pull-coord1-rate      = 0.002     ; nm/ps = 2 m/s pulling speed
pull-coord1-k         = 1000.0    ; kJ/mol/nm² spring constant

pull-nstxout  = 500              ; output coordinate every 500 steps
pull-nstfout  = 500              ; output force every 500 steps
"""

mdp_umbrella = """
; ============================================================
; umbrella.mdp — Umbrella sampling window (5 ns each)
; ============================================================
integrator      = md
nsteps          = 2500000        ; 5 ns
dt              = 0.002

nstenergy       = 1000
nstlog          = 1000
nstxout-compressed = 500

cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0

coulombtype     = PME
pme-order       = 4
fourierspacing  = 0.16

tcoupl          = V-rescale
tc-grps         = Protein  Non-Protein
tau_t           = 0.1    0.1
ref_t           = 300    300

pcoupl          = Parrinello-Rahman
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5

constraints     = h-bonds
constraint-algorithm = LINCS

; ===========================================================
; UMBRELLA SAMPLING PARAMETERS
; ===========================================================
pull            = yes
pull-ngroups    = 2
pull-ncoords    = 1

pull-group1-name = N_term
pull-group2-name = C_term

pull-coord1-type      = umbrella
pull-coord1-geometry  = distance
pull-coord1-groups    = 1 2
pull-coord1-dim       = Y Y Y
pull-coord1-rate      = 0.0           ; NO pulling — fixed window position
pull-coord1-k         = 1000.0        ; kJ/mol/nm² harmonic bias
pull-coord1-init      = WINDOW_POS    ; replace with actual window position in nm

pull-nstxout  = 500
pull-nstfout  = 500
"""

for name, content in [('pull.mdp', mdp_pull), ('umbrella.mdp', mdp_umbrella)]:
    with open(f'/tmp/{name}', 'w') as f:
        f.write(content)
    print(f'Wrote /tmp/{name}')
print()
print('pull.mdp key settings:')
for line in mdp_pull.strip().split('\n'):
    if 'pull-coord1' in line or line.startswith('; ==='):
        print(' ', line)

## 5. Running GROMACS

```bash
# ---- Setup ----
gmx pdb2gmx -f 2rvd.pdb -o protein.gro -water tip3p -ff amber99sb-ildn
gmx editconf -f protein.gro -o boxed.gro -c -d 1.2 -bt cubic
gmx solvate -cp boxed.gro -cs spc216.gro -o solv.gro -p topol.top
gmx genion -s ions.tpr -o ions.gro -p topol.top -pname NA -nname CL -neutral

# ---- Energy minimization ----
gmx grompp -f em.mdp -c ions.gro -p topol.top -o em.tpr
gmx mdrun -v -deffnm em

# ---- NVT equilibration ----
gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr
gmx mdrun -deffnm nvt

# ---- NPT equilibration ----
gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr
gmx mdrun -deffnm npt

# ---- SMD pulling ----
gmx grompp -f pull.mdp -c npt.gro -t npt.cpt -p topol.top -o pull.tpr
gmx mdrun -deffnm pull -pf pullf.xvg -px pullx.xvg

# ---- Extract umbrella windows ----
# (extract frames at ~0.1 nm spacing from pull trajectory)
gmx distance -f pull.xtc -s pull.tpr -select 'com of group N_term plus com of group C_term' -oall dist.xvg

# ---- Umbrella sampling (for each window i) ----
gmx grompp -f umbrella_win${i}.mdp -c conf_win${i}.gro -p topol.top -o us_win${i}.tpr
gmx mdrun -deffnm us_win${i} -pf usf_win${i}.xvg -px usx_win${i}.xvg

# ---- WHAM analysis ----
gmx wham -it tpr_files.dat -if pullf_files.dat -o pmf.xvg -hist hist.xvg -unit kJ
```

> **Note:** The code cells below use synthetic data mimicking real GROMACS output. All analysis runs without GROMACS installed.

## 6. Analysis: SMD Pulling Force and Work

During the constant-velocity pulling simulation, GROMACS outputs:
- `pullx.xvg`: end-to-end distance $\xi(t)$ vs. time
- `pullf.xvg`: pulling force $F(t)$ vs. time

The instantaneous work is:
$$W = \int_0^t F(t') \, \dot{\xi}(t') \, dt'$$

In [ ]:
def generate_pulling_data(n_points=5000, v_pull=0.002, dt=0.001, seed=42):
    """
    Generate synthetic SMD pulling data for chignolin (10-residue β-hairpin).
    v_pull: nm/ps,  dt: ps
    Returns: time(ps), distance(nm), force(kJ/mol/nm), work(kJ/mol)
    """
    np.random.seed(seed)
    t = np.arange(n_points) * dt
    
    # End-to-end distance: starts ~0.8 nm (folded), increases to ~3.0 nm (unfolded)
    xi = 0.80 + v_pull * t  # nm
    
    # Underlying PMF energy landscape with barriers (unfolding pathway)
    # Barrier 1 at ~1.3 nm (breaking hydrophobic core)
    # Barrier 2 at ~2.0 nm (breaking backbone H-bonds)
    def pmf_landscape(x):
        """PMF in kJ/mol as function of end-to-end distance (nm)"""
        pmf = (
            -20.0 * np.exp(-((x - 0.80)/0.15)**2)   # native state minimum
            + 18.0 * np.exp(-((x - 1.30)/0.20)**2)  # first barrier
            - 8.0  * np.exp(-((x - 1.65)/0.18)**2)  # intermediate
            + 12.0 * np.exp(-((x - 2.00)/0.22)**2)  # second barrier
            - 5.0  * np.exp(-((x - 2.40)/0.25)**2)  # second minimum
        )
        return pmf
    
    # Force = -dPMF/dx + harmonic spring + thermal noise
    k_spring = 1000.0   # kJ/mol/nm²
    
    # Virtual spring position (moves at v_pull)
    x_spring = 0.80 + v_pull * t
    
    # dPMF/dx (numerical)
    dpmf = np.gradient(pmf_landscape(xi), xi[1]-xi[0] if len(xi) > 1 else 0.001)
    
    # Force on spring = k*(x_spring - xi_actual) — but xi follows spring here
    # Simulate as: force ≈ -dPMF/dxi + noise
    noise = np.random.normal(0, 80, n_points)  # thermal fluctuations
    force = -dpmf + noise
    
    # Smooth the force for visualization
    force_smooth = savgol_filter(force, 201, 3)
    
    # Cumulative work W = ∫ F · v_pull dt
    work = np.cumsum(force_smooth * v_pull * dt)  # kJ/mol
    
    return t, xi, force, force_smooth, work

t_pull, xi_pull, F_raw, F_smooth, W_pull = generate_pulling_data()
print(f'SMD pulling data: {len(t_pull)} points, {t_pull[-1]:.1f} ps')
print(f'Distance range: {xi_pull[0]:.2f} – {xi_pull[-1]:.2f} nm')
print(f'Max force: {F_smooth.max():.1f} kJ/mol/nm = {F_smooth.max()/6.022:.1f} pN')
print(f'Total work: {W_pull[-1]:.1f} kJ/mol')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=False)

# Force vs distance
ax = axes[0]
ax.plot(xi_pull, F_raw, lw=0.4, color='lightblue', alpha=0.7, label='Raw force')
ax.plot(xi_pull, F_smooth, lw=2, color='royalblue', label='Smoothed force')
ax.axhline(0, ls='--', color='k', lw=0.8)
ax.set_xlabel('End-to-end distance ξ (nm)')
ax.set_ylabel('Force (kJ/mol/nm)')
ax.set_title('SMD Pulling Force — Chignolin Unfolding')
ax2 = ax.twinx()
ax2.set_ylabel('Force (pN)', color='gray')
ax2.set_ylim(np.array(ax.get_ylim()) / 6.022)
ax2.tick_params(colors='gray')
ax.legend(loc='upper left')

# Work vs distance
ax = axes[1]
ax.plot(xi_pull, W_pull, lw=2, color='darkorange')
ax.axhline(0, ls='--', color='k', lw=0.8)
ax.set_xlabel('End-to-end distance ξ (nm)')
ax.set_ylabel('Cumulative work W (kJ/mol)')
ax.set_title('Cumulative Work During SMD Pulling')
ax.text(xi_pull[-1]-0.3, W_pull[-1]+2, f'W = {W_pull[-1]:.0f} kJ/mol', ha='right', fontsize=10)

# Force vs time
ax = axes[2]
ax.plot(t_pull, F_smooth, lw=1.5, color='mediumpurple')
ax.axhline(0, ls='--', color='k', lw=0.8)
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Force (kJ/mol/nm)')
ax.set_title('Force vs. Time (constant-velocity pulling)')

plt.tight_layout()
plt.show()

## 7. Jarzynski Equality: Free Energy from Non-Equilibrium Work

The **Jarzynski equality** (1997) relates non-equilibrium work to equilibrium free energy:

$$e^{-\Delta F / k_B T} = \left\langle e^{-W / k_B T} \right\rangle_{\text{non-eq.}}$$

i.e., $\Delta F = -k_B T \ln \langle e^{-W/k_BT} \rangle$

This holds for **any** irreversible process connecting states A and B, as long as we average over many independent realizations. In practice:
- Run many pulling simulations with **different random velocity seeds**
- Collect work values $\{W_1, W_2, \ldots, W_N\}$ at each distance
- Apply the exponential average (or the Bennet acceptance ratio, BAR, for better convergence)

**Caveat:** The exponential average is dominated by rare low-work trajectories → slow convergence. Need many (~100+) repeats.

In [ ]:
def generate_work_trajectories(n_traj=50, n_points=5000, seed_base=0):
    """Generate N independent SMD work profiles."""
    works = []
    for i in range(n_traj):
        _, _, _, F_s, W = generate_pulling_data(n_points=n_points, seed=seed_base+i)
        works.append(W)
    return np.array(works)

print('Generating 50 independent SMD trajectories...')
W_matrix = generate_work_trajectories(n_traj=50)  # shape (50, 5000)

kBT = 2.479  # kJ/mol at 300 K (kB=8.314e-3 kJ/mol/K * 300 K)

# Jarzynski free energy at each distance point
# ΔF(ξ) = -kBT * ln( <exp(-W(ξ)/kBT)> )
dF_jarz = -kBT * np.log(np.mean(np.exp(-W_matrix / kBT), axis=0))

# Average work and std
W_mean = W_matrix.mean(axis=0)
W_std  = W_matrix.std(axis=0)

# Second-order cumulant approximation (more stable)
dF_2nd = W_mean - W_matrix.var(axis=0) / (2 * kBT)

print(f'Work at end (mean ± std): {W_mean[-1]:.1f} ± {W_std[-1]:.1f} kJ/mol')
print(f'ΔF (Jarzynski, end):      {dF_jarz[-1]:.1f} kJ/mol')
print(f'ΔF (2nd cumulant, end):   {dF_2nd[-1]:.1f} kJ/mol')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Work distributions at selected points
ax = axes[0]
xi_vals = np.linspace(0.80, 0.80 + 0.002*5000*0.001, 5000)
for frac, color, label in [(0.2,'tab:blue','ξ=1.08 nm'), (0.5,'tab:orange','ξ=1.80 nm'), (0.9,'tab:red','ξ=2.72 nm')]:
    idx = int(frac * 5000)
    w_vals = W_matrix[:, idx]
    ax.hist(w_vals, bins=15, alpha=0.55, color=color, edgecolor='w', label=label)
ax.set_xlabel('Work W (kJ/mol)')
ax.set_ylabel('Count')
ax.set_title('Work Distribution at Three Extension Points\n(50 SMD trajectories)')
ax.legend()

# Free energy vs distance
ax = axes[1]
ax.fill_between(xi_vals, W_mean - W_std, W_mean + W_std, alpha=0.15, color='gray', label='±1σ work')
ax.plot(xi_vals, W_mean,   lw=2,   color='gray',       label='⟨W⟩ (mean work)', ls='--')
ax.plot(xi_vals, dF_2nd,   lw=2.5, color='tab:orange', label='ΔF (2nd cumulant)')
ax.plot(xi_vals, dF_jarz,  lw=2,   color='tab:blue',   label='ΔF (Jarzynski)')
ax.axhline(0, ls=':', color='k', lw=0.8)
ax.set_xlabel('End-to-end distance ξ (nm)')
ax.set_ylabel('ΔF or W (kJ/mol)')
ax.set_title('Free Energy Profile via Jarzynski Equality\n(50 non-equilibrium SMD pulls)')
ax.legend(fontsize=9)
ax.text(0.97, 0.05, 'Note: Jarzynski convergence\nimproves with more trajectories',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

## 8. Umbrella Sampling

### 8.1 Principle

Rather than pulling rapidly (non-equilibrium), **umbrella sampling** restrains the system near a series of target values $\xi_i^0$ using a harmonic bias:

$$U_{\text{bias},i}(\xi) = \frac{k}{2}(\xi - \xi_i^0)^2$$

Each window samples a **biased** distribution $\rho_i^b(\xi)$. After simulation, WHAM removes the bias and reconstructs the unbiased PMF.

### 8.2 Window Setup

- Extract starting configurations from SMD trajectory at evenly-spaced $\xi$ values
- Typical: 20–40 windows, spacing ~0.1 nm, $k = 1000$ kJ/mol/nm²
- Each window: 5–10 ns production run with SHAKE + PME

### 8.3 WHAM: Weighted Histogram Analysis Method

WHAM iteratively solves for the unbiased distribution:

$$\rho(\xi) \propto \frac{\sum_i H_i(\xi)}{\sum_i n_i e^{(f_i - U_{bias,i}(\xi))/k_BT}}$$

where $f_i$ are free energy offsets solved self-consistently.

The PMF is then: $G(\xi) = -k_BT \ln \rho(\xi) + \text{const}$

In [ ]:
def generate_umbrella_data(xi0_list, k=1000.0, n_samples=10000, kBT=2.479, seed=0):
    """
    Generate umbrella sampling histograms for each window.
    The underlying PMF is the chignolin model.
    """
    np.random.seed(seed)
    
    def true_pmf(xi):
        return (
            0.0
            + 15.0 * np.exp(-((xi - 1.30)/0.20)**2)
            - 6.0  * np.exp(-((xi - 1.65)/0.18)**2)
            + 10.0 * np.exp(-((xi - 2.00)/0.22)**2)
            - 4.0  * np.exp(-((xi - 2.40)/0.25)**2)
            + 0.8 * (xi - 0.80)  # slow rising baseline
        )
    
    windows = []
    xi_fine = np.linspace(0.70, 3.20, 1000)
    
    for xi0 in xi0_list:
        # Biased probability ∝ exp(-(PMF + k/2*(xi-xi0)²)/kBT)
        V_total = true_pmf(xi_fine) + 0.5 * k * (xi_fine - xi0)**2
        prob = np.exp(-V_total / kBT)
        prob /= prob.sum()
        
        # Sample from this distribution
        samples = np.random.choice(xi_fine, size=n_samples, p=prob)
        samples += np.random.normal(0, 0.003, n_samples)  # small noise
        windows.append(samples)
    
    return windows, true_pmf

# 25 umbrella windows from 0.80 to 3.20 nm
xi0_list = np.linspace(0.80, 3.20, 25)
windows, true_pmf_fn = generate_umbrella_data(xi0_list, k=1000.0, n_samples=8000)

print(f'Generated {len(windows)} umbrella windows')
print(f'Window centers: {xi0_list[0]:.2f} to {xi0_list[-1]:.2f} nm')
print(f'Samples per window: {len(windows[0])}')

In [ ]:
# Simple WHAM implementation
def wham(windows, xi0_list, k=1000.0, kBT=2.479, n_bins=100, xi_min=0.70, xi_max=3.30, max_iter=5000, tol=1e-6):
    """
    Weighted Histogram Analysis Method.
    windows: list of sample arrays per window
    xi0_list: window centers (nm)
    k: spring constant (kJ/mol/nm²)
    Returns: bin centers, PMF (kJ/mol)
    """
    n_win = len(windows)
    edges = np.linspace(xi_min, xi_max, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    
    # Build histograms
    n_i = np.array([len(w) for w in windows], dtype=float)
    H = np.array([np.histogram(w, bins=edges)[0] for w in windows], dtype=float)  # (n_win, n_bins)
    
    # Bias matrix: U_bias[i, b] = 0.5*k*(center[b] - xi0[i])^2
    U_bias = 0.5 * k * (centers[np.newaxis, :] - xi0_list[:, np.newaxis])**2  # (n_win, n_bins)
    
    # WHAM iteration
    f = np.zeros(n_win)  # free energy offsets
    
    for iteration in range(max_iter):
        # Unbiased density
        denom = np.sum(n_i[:, np.newaxis] * np.exp((f[:, np.newaxis] - U_bias) / kBT), axis=0)  # (n_bins,)
        rho = H.sum(axis=0) / (denom + 1e-300)  # (n_bins,)
        
        # Update free energies
        f_new = -kBT * np.log(np.sum(rho[np.newaxis, :] * np.exp(-U_bias / kBT), axis=1) + 1e-300)
        f_new -= f_new[0]  # normalize
        
        if np.max(np.abs(f_new - f)) < tol:
            print(f'WHAM converged in {iteration+1} iterations')
            break
        f = f_new.copy()
    
    # Compute PMF
    rho[rho < 1e-100] = 1e-100
    pmf = -kBT * np.log(rho)
    pmf -= pmf.min()  # set minimum to 0
    
    return centers, pmf

xi_bins, pmf_wham = wham(windows, xi0_list, k=1000.0, kBT=2.479)

# True PMF for reference
xi_true = np.linspace(0.70, 3.30, 500)
pmf_true = true_pmf_fn(xi_true)
pmf_true -= pmf_true.min()

print(f'WHAM PMF: min={pmf_wham.min():.1f}, max={pmf_wham.max():.1f} kJ/mol')
print(f'Unfolding barrier (1st): ~{pmf_wham[(xi_bins>1.1)&(xi_bins<1.5)].max():.1f} kJ/mol')

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

# --- Histogram overlap ---
ax0 = fig.add_subplot(gs[0, 0])
cmap = plt.cm.viridis
for i, (samp, xi0) in enumerate(zip(windows, xi0_list)):
    color = cmap(i / len(xi0_list))
    counts, edges_h = np.histogram(samp, bins=80, range=(0.70, 3.30), density=True)
    centers_h = 0.5*(edges_h[:-1]+edges_h[1:])
    ax0.plot(centers_h, counts, color=color, lw=0.8, alpha=0.7)
ax0.set_xlabel('ξ (nm)')
ax0.set_ylabel('Probability density')
ax0.set_title('Umbrella Window Histograms\n(25 windows, color = ξ₀)')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0.80, 3.20))
sm.set_array([])
plt.colorbar(sm, ax=ax0, label='Window center (nm)')

# --- PMF comparison ---
ax1 = fig.add_subplot(gs[0, 1])
ax1.plot(xi_true, pmf_true, 'k--', lw=2, label='True PMF (input)', zorder=5)
ax1.plot(xi_bins, pmf_wham, 'b-',  lw=2.5, label='WHAM PMF (recovered)')
ax1.set_xlabel('ξ = end-to-end distance (nm)')
ax1.set_ylabel('PMF (kJ/mol)')
ax1.set_title('Potential of Mean Force: WHAM Recovery')
ax1.legend()
ax1.set_xlim(0.70, 3.30)

# Annotate states
ax1.annotate('Folded (F)', xy=(0.80, 0), xytext=(0.80, 6),
             arrowprops=dict(arrowstyle='->', color='green'),
             ha='center', color='green', fontsize=9)
ax1.annotate('Unfolded (U)', xy=(2.70, pmf_wham[np.argmin(np.abs(xi_bins-2.70))]),
             xytext=(2.70, pmf_wham[np.argmin(np.abs(xi_bins-2.70))]+6),
             arrowprops=dict(arrowstyle='->', color='red'),
             ha='center', color='red', fontsize=9)

# --- PMF comparison with SMD Jarzynski ---
ax2 = fig.add_subplot(gs[1, :])
# Interpolate Jarzynski result to same xi grid
xi_jarz_full = np.linspace(0.80, 0.80 + 0.002*5000*0.001, 5000)
from scipy.interpolate import interp1d
# Limit to where both exist
xi_overlap = xi_jarz_full[(xi_jarz_full >= 0.80) & (xi_jarz_full <= 3.20)]
dF_jarz_interp = dF_jarz[(xi_jarz_full >= 0.80) & (xi_jarz_full <= 3.20)]
dF_jarz_interp -= dF_jarz_interp.min()

ax2.plot(xi_true, pmf_true, 'k--', lw=2, label='True PMF', zorder=5)
ax2.plot(xi_bins, pmf_wham,       lw=2.5, color='royalblue',  label='Umbrella + WHAM (equil.)')
ax2.plot(xi_overlap, dF_jarz_interp, lw=2, color='darkorange', alpha=0.8, label='SMD + Jarzynski (non-equil., 50 traj.)')
ax2.set_xlabel('ξ = end-to-end distance (nm)')
ax2.set_ylabel('PMF / ΔF (kJ/mol)')
ax2.set_title('Free Energy Profile: Umbrella+WHAM vs. SMD+Jarzynski')
ax2.legend()
ax2.set_xlim(0.70, 3.30)
ax2.text(0.02, 0.95,
         'WHAM is more accurate; Jarzynski requires many trajectories for convergence',
         transform=ax2.transAxes, ha='left', va='top', fontsize=9, color='gray')

plt.suptitle('Chignolin Unfolding — PMF Analysis', fontsize=14, y=1.01)
plt.show()

## 9. GROMACS Energy Analysis

After the simulation, use `gmx energy` to extract thermodynamic data:

```bash
gmx energy -f npt.edr -o energy.xvg
# Select: 10 12 15 (Potential, Kinetic, Temperature)
```

The `.xvg` file is plain ASCII — we can parse it with Python.

In [ ]:
def parse_xvg(filename):
    """
    Parse a GROMACS .xvg file.
    Returns: (times, data_array) ignoring comment lines starting with # or @
    """
    times, data = [], []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or line.startswith('@'):
                continue
            vals = list(map(float, line.split()))
            times.append(vals[0])
            data.append(vals[1:])
    return np.array(times), np.array(data)

# Generate synthetic GROMACS .xvg energy file
def write_synthetic_xvg(filename, n_frames=500):
    np.random.seed(10)
    t = np.arange(n_frames) * 2.0  # ps
    # Temperature: equilibrates from 310 K
    T = 300 + 10*np.exp(-t/50) + np.random.normal(0, 3, n_frames)
    # Potential energy per molecule (kJ/mol, ~ -26000 for ~1000 water + protein)
    Epot = -26000 + 150*np.exp(-t/40) + np.random.normal(0, 50, n_frames)
    # Kinetic energy
    Ekin = 3/2 * 8.314e-3 * T * 1000  # rough estimate for 1000 atoms
    
    with open(filename, 'w') as f:
        f.write('# GROMACS energy output (synthetic)\n')
        f.write('@ title "Energy"\n')
        f.write('@ xaxis label "Time (ps)"\n')
        f.write('@ yaxis label "Energy (kJ/mol)"\n')
        f.write('@ s0 legend "Potential"\n')
        f.write('@ s1 legend "Kinetic"\n')
        f.write('@ s2 legend "Temperature (K)"\n')
        for i in range(n_frames):
            f.write(f'{t[i]:.3f}  {Epot[i]:.3f}  {Ekin[i]:.3f}  {T[i]:.3f}\n')
    return t, Epot, Ekin, T

t_en, Epot, Ekin, T_traj = write_synthetic_xvg('/tmp/energy.xvg')
t_read, data_read = parse_xvg('/tmp/energy.xvg')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ['Potential Energy (kJ/mol)', 'Kinetic Energy (kJ/mol)', 'Temperature (K)']
colors = ['tab:blue', 'tab:orange', 'tab:red']
refs   = [None, None, 300]
for i, (ax, lbl, col, ref) in enumerate(zip(axes, labels, colors, refs)):
    ax.plot(t_read, data_read[:, i], lw=1, color=col, alpha=0.8)
    if ref:
        ax.axhline(ref, ls='--', color='k', lw=1, label=f'Target {ref} K')
        ax.legend(fontsize=9)
    ax.set_xlabel('Time (ps)')
    ax.set_ylabel(lbl)
    ax.set_title(lbl.split(' (')[0])

plt.suptitle('GROMACS NPT Energy File Analysis', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()
print(f'Average T (last 400 frames): {T_traj[100:].mean():.1f} ± {T_traj[100:].std():.1f} K')

## 10. Complete Workflow Summary

### Flowchart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
ax.set_xlim(0, 10)
ax.set_ylim(0, 24)
ax.axis('off')

def draw_box(ax, x, y, w, h, text, color='#E8F4FD', textsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='#2C6FAC', lw=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=textsize, wrap=True,
            multialignment='center')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2C6FAC', lw=1.5))

steps = [
    (3.0, 22.5, 4.0, 1.0, 'protein.pdb\n(download from RCSB/PDB)', '#D5E8D4'),
    (3.0, 20.5, 4.0, 1.0, 'gmx pdb2gmx\nAssign AMBER99SB-ILDN + TIP3P', '#E8F4FD'),
    (3.0, 18.5, 4.0, 1.0, 'gmx editconf + solvate + genion\nAdd water box + ions', '#E8F4FD'),
    (3.0, 16.5, 4.0, 1.0, 'Energy Minimization\n(em.mdp, steepest descent)', '#FFF2CC'),
    (3.0, 14.5, 4.0, 1.0, 'NVT Equilibration (100 ps)\n(nvt.mdp, V-rescale, T=300 K)', '#FFF2CC'),
    (3.0, 12.5, 4.0, 1.0, 'NPT Equilibration (100 ps)\n(npt.mdp, Parrinello-Rahman, P=1 bar)', '#FFF2CC'),
    (3.0, 10.0, 4.0, 1.0, 'SMD Pulling (5 ns)\n(pull.mdp, v=0.002 nm/ps, k=1000)', '#F8CECC'),
    (3.0,  7.5, 4.0, 1.5, 'Extract Umbrella Windows\n(frames at Δξ=0.1 nm spacing)\n25–40 windows', '#E8F4FD'),
    (3.0,  5.5, 4.0, 1.0, 'Umbrella Sampling\n(5 ns × N windows)', '#F8CECC'),
    (3.0,  3.5, 4.0, 1.0, 'gmx wham → PMF\n(WHAM analysis)', '#D5E8D4'),
    (3.0,  1.5, 4.0, 1.0, 'Free Energy Profile ΔG(ξ)\n+ Error Bars (bootstrap)', '#D5E8D4'),
]

for x, y, w, h, text, col in steps:
    draw_box(ax, x, y, w, h, text, color=col)

# Arrows
for i in range(len(steps)-1):
    x1 = steps[i][0] + steps[i][2]/2
    y1 = steps[i][1]
    x2 = steps[i+1][0] + steps[i+1][2]/2
    y2 = steps[i+1][1] + steps[i+1][3]
    arrow(ax, x1, y1, x2, y2)

# Side annotations
ax.text(7.5, 10.5, 'pullx.xvg\npullf.xvg', fontsize=8, color='#C55A11', ha='left')
ax.text(7.5,  6.0, 'conf_win_i.gro\n(N windows)', fontsize=8, color='#C55A11', ha='left')
ax.text(7.5,  6.5, '←', fontsize=14, color='#C55A11')
ax.text(7.5, 10.5, '←', fontsize=14, color='#C55A11')

ax.set_title('GROMACS Protein Unfolding Workflow\n(Chignolin / AMBER99SB-ILDN / TIP3P)',
             fontsize=12, pad=10)
plt.tight_layout()
plt.show()

## 11. Key Takeaways

### Methods Summary

| Method | Bias type | Equilibrium? | Output | Cost |
|--------|-----------|-------------|--------|------|
| Unbiased MD | None | Yes | Trajectory | Very high for rare events |
| Steered MD | Constant-velocity harmonic | No | F(t), W(ξ) | Low |
| Jarzynski (SMD) | Non-eq. work average | ~Yes | ΔF(ξ) | Medium (many repeats) |
| Umbrella sampling | Fixed harmonic restraint | Yes | PMF via WHAM | High |

### Physical Insights from Chignolin Unfolding

1. **First barrier (~1.3 nm):** Breaking the hydrophobic core contacts
2. **Intermediate state (~1.65 nm):** Partially extended, some H-bonds intact
3. **Second barrier (~2.0 nm):** Breaking backbone H-bonds of the β-hairpin
4. **Unfolded basin (~2.5+ nm):** Random coil, $\Delta G_{fold} \approx -5$ kcal/mol

### Best Practices

- Always equilibrate in stages: EM → NVT (restrained) → NPT (restrained) → NPT (unrestrained)
- Check histogram overlap in umbrella sampling — add more windows if gaps exist
- Use bootstrap error analysis (`gmx wham -bs`) to estimate PMF uncertainty
- Validate with experimental data: $T_m$, $\Delta H_{unfold}$ from DSC, end-to-end distance from FRET

### References

1. Lindorff-Larsen, K. *et al.* Proteins **2010**, 78, 1950 (AMBER99SB-ILDN)
2. Honda, S. *et al.* J. Am. Chem. Soc. **2008**, 130, 15327 (Chignolin CLN025)
3. Jarzynski, C. Phys. Rev. Lett. **1997**, 78, 2690 (Jarzynski equality)
4. Kumar, S. *et al.* J. Comput. Chem. **1992**, 13, 1011 (WHAM)
5. GROMACS Documentation: https://manual.gromacs.org
6. Lemkul, J. A. Living J. Comput. Mol. Sci. **2019** (GROMACS tutorial)